In [ ]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from lightgbm import LGBMClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, SMOTENC

# ========== 0) Assume you already split your dataset ==========
# You should have these already prepared in your workflow:
# X_train_raw, X_test_raw, y_train, y_test
# categorical_cols, numeric_cols

# ====== 1) LightGBM pipeline ======
lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbose=-1  # Suppress warnings
)

preprocessor_lgbm = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

lgbm_pipe = ImbPipeline(steps=[
    ('preprocessor', preprocessor_lgbm),
    ('smote', SMOTE(random_state=42, sampling_strategy='minority')),  # Changed to 'auto'
    ('classifier', lgbm)
])

# ====== 2) TabNet pipeline with proper categorical handling ======
# Calculate categorical dimensions and indices
cat_dims = [X_train_raw[col].nunique() for col in categorical_cols]
cat_idxs = list(range(len(categorical_cols)))

preprocessor_tabnet = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# SMOTENC for mixed data types
smotenc = SMOTENC(
    categorical_features=cat_idxs,  # Only categorical indices
    sampling_strategy='auto',
    random_state=42
)

tabnet_clf = TabNetClassifier(
    n_d=32, n_a=32, n_steps=5,
    gamma=1.5,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=3,
    n_independent=2, 
    n_shared=2,
    momentum=0.02,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
    verbose=0
)

# Function to convert to float32 (required for TabNet)
def to_float32(X):
    return X.astype(np.float32)

float32_transformer = FunctionTransformer(to_float32, validate=False)

tabnet_pipe = ImbPipeline(steps=[
    ('preprocessor_tabnet', preprocessor_tabnet),
    ('smotenc', smotenc),
    ('to_float32', float32_transformer),
    ('classifier', tabnet_clf)
])

# ====== 3) Wrapper class for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    """Wrapper to make imblearn pipelines compatible with sklearn ensembles"""
    
    def __init__(self, pipeline):
        self.pipeline = pipeline
        
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
        
    def predict(self, X):
        return self.pipeline.predict(X)
        
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# Wrap the pipelines
lgbm_wrapped = ImblearnWrapper(lgbm_pipe)
tabnet_wrapped = ImblearnWrapper(tabnet_pipe)

# ====== 4) Individual model training and evaluation ======
individual_models = {
    'LightGBM': lgbm_pipe,
    'TabNet': tabnet_pipe,
}

print("🔥 Training Individual Models...")
individual_results = []

for name, model in individual_models.items():
    print(f"\n🚀 Training: {name}")
    try:
        model.fit(X_train_raw, y_train)
        
        y_pred = model.predict(X_test_raw)
        y_proba = model.predict_proba(X_test_raw)
        
        # Calculate metrics
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        
        individual_results.append({
            'Model': name,
            'Precision (Macro)': precision,
            'Recall (Macro)': recall,
            'F1 Score (Macro)': f1,
            'Accuracy': accuracy,
            'AUC': auc
        })
        
        print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
    except Exception as e:
        print(f"❌ Error training {name}: {str(e)}")


# ====== 6) Manual ensemble methods (only if individual models trained successfully) ======
if len(individual_results) == 2:
    print("\n⚡ Creating Manual Ensembles...")
    
    # Get probabilities from individual models
    lgbm_proba = individual_models['LightGBM'].predict_proba(X_test_raw)[:, 1]
    tabnet_proba = individual_models['TabNet'].predict_proba(X_test_raw)[:, 1]
    
    manual_results = []
    
    # Threshold tuning
    thresh_lgbm, thresh_tabnet = 0.5, 0.6
    y_pred_thresh = ((lgbm_proba > thresh_lgbm) | (tabnet_proba > thresh_tabnet)).astype(int)
    
    # Calculate metrics for threshold ensemble
    precision_thresh = precision_score(y_test, y_pred_thresh, average='macro', zero_division=0)
    recall_thresh = recall_score(y_test, y_pred_thresh, average='macro', zero_division=0)
    f1_thresh = f1_score(y_test, y_pred_thresh, average='macro', zero_division=0)
    accuracy_thresh = accuracy_score(y_test, y_pred_thresh)
    
    manual_results.append({
        'Model': 'Threshold Ensemble',
        'Precision (Macro)': precision_thresh,
        'Recall (Macro)': recall_thresh,
        'F1 Score (Macro)': f1_thresh,
        'Accuracy': accuracy_thresh,
        'AUC': None  # Not applicable for threshold method
    })
    
    print(f"✅ Threshold Ensemble - Accuracy: {accuracy_thresh:.4f}, F1: {f1_thresh:.4f}")
    
    # Weighted average
    alpha, beta = 0.7, 0.3
    ensemble_proba = alpha * lgbm_proba + beta * tabnet_proba
    y_pred_weighted = (ensemble_proba > 0.5).astype(int)
    
    # Calculate AUC for weighted ensemble
    auc_weighted = roc_auc_score(y_test, ensemble_proba)
    precision_weighted = precision_score(y_test, y_pred_weighted, average='macro', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred_weighted, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred_weighted, average='macro', zero_division=0)
    accuracy_weighted = accuracy_score(y_test, y_pred_weighted)
    
    manual_results.append({
        'Model': 'Weighted Average',
        'Precision (Macro)': precision_weighted,
        'Recall (Macro)': recall_weighted,
        'F1 Score (Macro)': f1_weighted,
        'Accuracy': accuracy_weighted,
        'AUC': auc_weighted
    })
    
    print(f"✅ Weighted Average - Accuracy: {accuracy_weighted:.4f}, F1: {f1_weighted:.4f}, AUC: {auc_weighted:.4f}")
    
    # Hybrid rules
    y_pred_hybrid = []
    for p_l, p_t in zip(lgbm_proba, tabnet_proba):
        if p_t > 0.7 and p_l > 0.4:
            y_pred_hybrid.append(1)
        elif p_t > 0.8:
            y_pred_hybrid.append(1)
        elif p_l > 0.6:
            y_pred_hybrid.append(1)
        else:
            y_pred_hybrid.append(0)
    
    y_pred_hybrid = np.array(y_pred_hybrid)
    precision_hybrid = precision_score(y_test, y_pred_hybrid, average='macro', zero_division=0)
    recall_hybrid = recall_score(y_test, y_pred_hybrid, average='macro', zero_division=0)
    f1_hybrid = f1_score(y_test, y_pred_hybrid, average='macro', zero_division=0)
    accuracy_hybrid = accuracy_score(y_test, y_pred_hybrid)
    
    manual_results.append({
        'Model': 'Hybrid Rules',
        'Precision (Macro)': precision_hybrid,
        'Recall (Macro)': recall_hybrid,
        'F1 Score (Macro)': f1_hybrid,
        'Accuracy': accuracy_hybrid,
        'AUC': None  # Not applicable for rule-based method
    })
    
    print(f"✅ Hybrid Rules - Accuracy: {accuracy_hybrid:.4f}, F1: {f1_hybrid:.4f}")

# ====== 7) Compile and save all results ======
all_results = individual_results
if 'manual_results' in locals():
    all_results.extend(manual_results)

results_df = pd.DataFrame(all_results)

# Display results
print("\n📊 FINAL RESULTS SUMMARY:")
print("="*80)
print(results_df.to_string(index=False, float_format='%.4f'))

# Save to CSV
results_df.to_csv("lgbm_tabnet_ensemble_results_v2_minority_SMOTE_tabnetsupport.csv", index=False)
print(f"\n✅ Results exported to: lgbm_tabnet_ensemble_results_v2_minoritySMOTE.csv")

# Find best model
best_f1_idx = results_df['F1 Score (Macro)'].idxmax()
best_model = results_df.loc[best_f1_idx, 'Model']
best_f1 = results_df.loc[best_f1_idx, 'F1 Score (Macro)']

print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")